In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def pretty_matrix_as_grid(v: np.ndarray, nrow: int, ncol: int, decimals: int = 2):
    grid = np.asarray(v, dtype=float).reshape(nrow, ncol)
    with np.printoptions(precision=decimals, suppress=True):
        print(grid)

def action_arrows(pi_det: np.ndarray, nrow: int, ncol: int, arrows: dict[int, str]):
    out = []
    for r in range(nrow):
        row = []
        for c in range(ncol):
            s = r * ncol + c
            row.append(arrows.get(int(pi_det[s]), '?'))
        out.append(' '.join(row))
    print('\n'.join(out))

class _Discrete:
    def __init__(self, n: int):
        self.n = int(n)

class PModelEnv:
    def __init__(self, P, start_state: int = 0, seed: int = 0):
        self.P = P
        self.nS = len(P)
        s0 = next(iter(P))
        self.nA = len(P[s0])
        self.action_space = _Discrete(self.nA)
        self.observation_space = _Discrete(self.nS)
        self.start_state = int(start_state)
        self.rng = np.random.default_rng(seed)
        self.s = self.start_state

    def reset(self, seed: int | None = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.s = self.start_state
        return int(self.s), {}

    def step(self, a: int):
        a = int(a)
        outcomes = self.P[self.s][a]
        ps = np.array([o[0] for o in outcomes], dtype=float)
        idx = int(self.rng.choice(len(outcomes), p=ps/ps.sum()))
        p, s2, r, terminated = outcomes[idx]
        self.s = int(s2)
        return int(s2), float(r), bool(terminated), False, {}

def build_frozenlake_P(desc, is_slippery: bool = False):
    desc = np.asarray([list(row) for row in desc], dtype="<U1")
    nrow, ncol = desc.shape
    nS, nA = nrow * ncol, 4
    LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
    moves = {LEFT:(0,-1), DOWN:(1,0), RIGHT:(0,1), UP:(-1,0)}
    def to_s(r,c): return r*ncol+c
    def step_from(r,c,a):
        dr,dc = moves[a]; r2,c2 = r+dr, c+dc
        if r2<0 or r2>=nrow or c2<0 or c2>=ncol: r2,c2=r,c
        return r2,c2

    P = {s: {a: [] for a in range(nA)} for s in range(nS)}
    for r in range(nrow):
        for c in range(ncol):
            s = to_s(r,c); tile = desc[r,c]
            if tile in ("H","G"):
                for a in range(nA):
                    P[s][a] = [(1.0, s, 0.0, True)]
                continue
            for a in range(nA):
                if is_slippery:
                    candidates = [(a-1)%4, a, (a+1)%4]; probs = [1/3,1/3,1/3]
                else:
                    candidates = [a]; probs = [1.0]
                outcomes=[]
                for a_real,p in zip(candidates,probs):
                    r2,c2 = step_from(r,c,a_real)
                    s2 = to_s(r2,c2); tile2 = desc[r2,c2]
                    term = tile2 in ("H","G")
                    rwd = 1.0 if tile2=="G" else 0.0
                    outcomes.append((float(p), int(s2), float(rwd), bool(term)))
                merged={}
                for p,s2,rwd,term in outcomes:
                    key=(s2,rwd,term); merged[key]=merged.get(key,0.0)+p
                P[s][a] = [(p,s2,rwd,term) for (s2,rwd,term),p in merged.items()]
    return P, nS, nA, nrow, ncol, desc

desc4 = ["SFFF","FHFH","FFFH","HFFG"]
P_fl_det, nS, nA, nrow, ncol, desc = build_frozenlake_P(desc4, is_slippery=False)
P_fl_slip, _, _, _, _, _ = build_frozenlake_P(desc4, is_slippery=True)

env = PModelEnv(P_fl_det, start_state=0, seed=0)
env_slip = PModelEnv(P_fl_slip, start_state=0, seed=0)

arrows_fl = {0:"←",1:"↓",2:"→",3:"↑"}
print("\n".join(desc4))
print("n_states =", nS, "n_actions =", nA)


TypeError: 'type' object is not subscriptable

In [2]:
# %% [markdown]
# # RL Lab (6h): Model-Free RL po DP — Monte Carlo, TD, SARSA/Q-learning + krótki preview DQN
#
# Ten notebook jest zaprojektowany jako **kontynuacja Ch04 (Dynamic Programming)**.
# W Ch04 mieliśmy model środowiska `P[s][a]`. Teraz przechodzimy do metod **model-free**,
# które uczą się z *doświadczenia* (trajektorii), a nie z jawnego modelu.
#
# **Jak pracować z notebookiem:**
# - W sekcjach „Ćwiczenie” są komórki z `TODO` — studenci uzupełniają implementacje.
# - W sekcjach „Rozwiązanie (dla prowadzącego)” są kompletne implementacje (możesz je usunąć w wersji studenckiej).
#
# TODO: WSTAW RYSUNEK (Sutton): full backup vs sample backup

# %% [markdown]
# ## Cele zajęć (6h)
# Po tych zajęciach student powinien umieć:
# 1) Wyjaśnić różnicę: **DP = full backup** vs **MC/TD = sample backup**  
# 2) Zaimplementować MC prediction i MC control (on-policy)  
# 3) Zaimplementować TD(0) prediction oraz SARSA i rozumieć różnicę SARSA vs Q-learning  
# 4) Zinterpretować wykresy i wyciągnąć wnioski  
# 5) Zobaczyć, skąd bierze się DQN (preview): replay buffer + target network  
#
# W praktyce pracujemy na FrozenLake (det i slippery).

# %% [markdown]
# ---
# ## 0. Setup: wspólne narzędzia + środowisko z modelu `P[s][a]`
#
# W Ch04 budowaliśmy środowiska jako model:
# `P[s][a] = [(p, s2, r, terminated), ...]`.
#
# Teraz użyjemy tego samego `P` jako „symulatora”: będziemy losować przejścia i generować epizody.
# Dzięki temu uczymy się **model-free**, ale bez instalacji Gym.

# %%
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def pretty_matrix_as_grid(v: np.ndarray, nrow: int, ncol: int, decimals: int = 2):
    grid = np.asarray(v, dtype=float).reshape(nrow, ncol)
    with np.printoptions(precision=decimals, suppress=True):
        print(grid)

def action_arrows(pi_det: np.ndarray, nrow: int, ncol: int, arrows: dict[int, str]):
    out = []
    for r in range(nrow):
        row = []
        for c in range(ncol):
            s = r * ncol + c
            row.append(arrows.get(int(pi_det[s]), '?'))
        out.append(' '.join(row))
    print('\n'.join(out))

class _Discrete:
    def __init__(self, n: int):
        self.n = int(n)

class PModelEnv:
    """Minimalne środowisko typu Gym, ale oparte o model P[s][a]."""
    def __init__(self, P, start_state: int = 0, seed: int = 0):
        self.P = P
        self.nS = len(P)
        s0 = next(iter(P))
        self.nA = len(P[s0])
        self.action_space = _Discrete(self.nA)
        self.observation_space = _Discrete(self.nS)
        self.start_state = int(start_state)
        self.rng = np.random.default_rng(seed)
        self.s = self.start_state

    def reset(self, seed: int | None = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.s = self.start_state
        return int(self.s), {}

    def step(self, a: int):
        a = int(a)
        outcomes = self.P[self.s][a]
        ps = np.array([o[0] for o in outcomes], dtype=float)
        idx = int(self.rng.choice(len(outcomes), p=ps/ps.sum()))
        p, s2, r, terminated = outcomes[idx]
        self.s = int(s2)
        return int(s2), float(r), bool(terminated), False, {}

def build_frozenlake_P(desc, is_slippery: bool = False):
    """FrozenLake w formacie P[s][a] bez Gym."""
    desc = np.asarray([list(row) for row in desc], dtype="<U1")
    nrow, ncol = desc.shape
    nS, nA = nrow * ncol, 4

    LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
    moves = {LEFT:(0,-1), DOWN:(1,0), RIGHT:(0,1), UP:(-1,0)}

    def to_s(r,c): return r*ncol+c
    def step_from(r,c,a):
        dr,dc = moves[a]
        r2,c2 = r+dr, c+dc
        if r2<0 or r2>=nrow or c2<0 or c2>=ncol:
            r2,c2 = r,c
        return r2,c2

    P = {s: {a: [] for a in range(nA)} for s in range(nS)}

    for r in range(nrow):
        for c in range(ncol):
            s = to_s(r,c)
            tile = desc[r,c]

            # terminale absorbujące
            if tile in ("H","G"):
                for a in range(nA):
                    P[s][a] = [(1.0, s, 0.0, True)]
                continue

            for a in range(nA):
                if is_slippery:
                    candidates = [(a-1)%4, a, (a+1)%4]
                    probs = [1/3, 1/3, 1/3]
                else:
                    candidates = [a]
                    probs = [1.0]

                outcomes = []
                for a_real, p in zip(candidates, probs):
                    r2,c2 = step_from(r,c,a_real)
                    s2 = to_s(r2,c2)
                    tile2 = desc[r2,c2]
                    term = tile2 in ("H","G")
                    rwd = 1.0 if tile2 == "G" else 0.0
                    outcomes.append((float(p), int(s2), float(rwd), bool(term)))

                merged = {}
                for p, s2, rwd, term in outcomes:
                    key = (s2, rwd, term)
                    merged[key] = merged.get(key, 0.0) + p

                P[s][a] = [(p, s2, rwd, term) for (s2, rwd, term), p in merged.items()]

    return P, nS, nA, nrow, ncol, desc

desc4 = ["SFFF","FHFH","FFFH","HFFG"]
P_fl_det, nS, nA, nrow, ncol, desc = build_frozenlake_P(desc4, is_slippery=False)
P_fl_slip, _, _, _, _, _ = build_frozenlake_P(desc4, is_slippery=True)

env = PModelEnv(P_fl_det, start_state=0, seed=0)
env_slip = PModelEnv(P_fl_slip, start_state=0, seed=0)

arrows_fl = {0:"←",1:"↓",2:"→",3:"↑"}

print("FrozenLake map:")
print("\n".join(desc4))
print("n_states =", nS, "n_actions =", nA)

# %% [markdown]
# ---
# ## 1. Referencja z DP (z Ch04): Value Iteration jako „ground truth”
# W model-free często nie znamy `P`, ale tu (dla dydaktyki) możemy policzyć `v*` i `π*`
# i porównać z tym, co wyjdzie z MC/TD.
#
# TODO: WSTAW PSEUDOKOD (Sutton): Value Iteration

# %%
def greedy_policy_from_v(P, v: np.ndarray, gamma: float = 0.99) -> np.ndarray:
    nS = v.shape[0]
    s0 = next(iter(P))
    nA = len(P[s0])
    pi_det = np.zeros(nS, dtype=int)

    for s in range(nS):
        best_a = 0
        best_q = -np.inf
        for a in range(nA):
            q = 0.0
            for (p, s2, r, terminated) in P[s][a]:
                if terminated:
                    q += float(p) * float(r)
                else:
                    q += float(p) * (float(r) + gamma * float(v[int(s2)]))
            if q > best_q:
                best_q = q
                best_a = a
        pi_det[s] = best_a
    return pi_det

def value_iteration(P, gamma: float = 0.99, theta: float = 1e-10, max_iters: int = 100_000):
    nS = len(P)
    s0 = next(iter(P))
    nA = len(P[s0])
    v = np.zeros(nS, dtype=float)

    for _ in range(max_iters):
        delta = 0.0
        for s in range(nS):
            v_old = v[s]
            best_q = -np.inf
            for a in range(nA):
                q = 0.0
                for (p, s2, r, terminated) in P[s][a]:
                    if terminated:
                        q += float(p) * float(r)
                    else:
                        q += float(p) * (float(r) + gamma * float(v[int(s2)]))
                best_q = max(best_q, q)
            v[s] = best_q
            delta = max(delta, abs(v_old - v[s]))
        if delta < theta:
            break

    pi_det = greedy_policy_from_v(P, v, gamma=gamma)
    return pi_det, v

gamma = 0.99
pi_star_det, v_star_det = value_iteration(P_fl_det, gamma=gamma)
pi_star_slip, v_star_slip = value_iteration(P_fl_slip, gamma=gamma)

print("V* (det):")
pretty_matrix_as_grid(v_star_det, nrow, ncol, decimals=2)
print("\npi* (det):")
action_arrows(pi_star_det, nrow, ncol, arrows_fl)

# %% [markdown]
# ---
# ## 2. Helpery do MC/TD (dane + wykresy)
# To nie są zadania — to narzędzia.
#
# TODO: WSTAW RYSUNEK (Sutton): definicja returnu G_t

# %%
def epsilon_greedy_action(q_s: np.ndarray, eps: float, rng: np.random.Generator) -> int:
    if rng.random() < eps:
        return int(rng.integers(0, len(q_s)))
    return int(np.argmax(q_s))

def generate_episode_det_policy(env: PModelEnv, pi_det: np.ndarray, max_steps: int = 200, seed: int | None = None):
    s, _ = env.reset(seed=seed)
    episode = []
    for _ in range(max_steps):
        a = int(pi_det[s])
        s2, r, done, trunc, _ = env.step(a)
        episode.append((s, a, r))
        if done or trunc:
            break
        s = s2
    return episode

def generate_episode_eps_greedy(env: PModelEnv, Q: np.ndarray, eps: float, rng: np.random.Generator, max_steps: int = 200):
    s, _ = env.reset()
    episode = []
    for _ in range(max_steps):
        a = epsilon_greedy_action(Q[s], eps, rng)
        s2, r, done, trunc, _ = env.step(a)
        episode.append((s, a, r))
        if done or trunc:
            break
        s = s2
    return episode

def moving_average(x, window: int = 200):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    w = np.ones(window) / window
    return np.convolve(x, w, mode="valid")

# %% [markdown]
# ---
# ## 3. Monte Carlo prediction (first-visit): estymacja Vπ z epizodów
#
# **MC prediction** liczy `Vπ` na podstawie pełnych zwrotów `G`.
# Wersja first-visit aktualizuje stan tylko przy pierwszym wystąpieniu w epizodzie.
#
# TODO: WSTAW PSEUDOKOD (Sutton): first-visit MC prediction

# %% [markdown]
# ### Ćwiczenie 1 — MC prediction (first-visit)
# Zaimplementuj:
# - `V` i `N` (liczniki odwiedzin)
# - return `G` liczony od końca epizodu
# - first-visit update
# - średnia inkrementalna: `V[s] += (G - V[s]) / N[s]`

# %%
def mc_prediction_first_visit(env: PModelEnv, pi_det: np.ndarray,
                              gamma: float = 0.99, episodes: int = 20_000,
                              seed: int = 0, max_steps: int = 200):
    """
    TODO:
    - V = zeros(nS), N = zeros(nS)
    - for ep:
        episode = generate_episode_det_policy(...)
        G = 0
        visited = set()
        for t od końca:
            G = r + gamma*G
            jeśli first-visit:
                N[s] += 1
                V[s] += (G - V[s]) / N[s]
    """
    raise NotImplementedError("TODO: mc_prediction_first_visit")

# %% [markdown]
# #### Rozwiązanie (dla prowadzącego) — Ćwiczenie 1
# (Usuń w wersji studenckiej)

# %%
def mc_prediction_first_visit(env: PModelEnv, pi_det: np.ndarray,
                              gamma: float = 0.99, episodes: int = 20_000,
                              seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    V = np.zeros(nS, dtype=float)
    N = np.zeros(nS, dtype=int)
    rng = np.random.default_rng(seed)

    for ep in range(episodes):
        ep_seed = int(rng.integers(0, 10_000_000))
        episode = generate_episode_det_policy(env, pi_det, max_steps=max_steps, seed=ep_seed)

        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = float(r) + gamma * G
            if s in visited:
                continue
            visited.add(s)
            N[s] += 1
            V[s] += (G - V[s]) / N[s]
    return V

# %% [markdown]
# ### Demo: MC prediction dla π* (det)
# Porównaj do DP (VI).

# %%
gamma = 0.99
V_mc = mc_prediction_first_visit(env, pi_star_det, gamma=gamma, episodes=30_000, seed=0)

print("MC V_pi (pi* det):")
pretty_matrix_as_grid(V_mc, nrow, ncol, decimals=2)

print("\nDP V* (det):")
pretty_matrix_as_grid(v_star_det, nrow, ncol, decimals=2)

print("\nmax |MC - DP| =", float(np.max(np.abs(V_mc - v_star_det))))

# %% [markdown]
# ---
# ## 4. Monte Carlo control (on-policy): uczymy Q i politykę
#
# Uczymy `Q(s,a)` z epizodów generowanych przez ε-greedy względem Q.
# Na końcu bierzemy politykę zachłanną: `pi_det = argmax_a Q(s,a)`.
#
# TODO: WSTAW PSEUDOKOD (Sutton): on-policy first-visit MC control

# %% [markdown]
# ### Ćwiczenie 2 — MC control (on-policy, first-visit)
# Zaimplementuj `mc_control_on_policy`.
# Zwróć: `Q`, `pi` (det) i `success_curve` (0/1 czy epizod dotarł do celu).

# %%
def mc_control_on_policy(env: PModelEnv,
                         gamma: float = 0.99, episodes: int = 50_000,
                         eps: float = 0.2, seed: int = 0, max_steps: int = 200):
    """
    TODO:
    - Q = zeros((nS,nA)), N = zeros((nS,nA))
    - for ep:
        episode = generate_episode_eps_greedy(...)
        policz G od końca
        first-visit update dla (s,a)
        success_curve.append(1 jeśli w epizodzie r==1 else 0)
    - pi_det = argmax(Q, axis=1)
    - return dict(Q=..., pi=..., success_curve=...)
    """
    raise NotImplementedError("TODO: mc_control_on_policy")

# %% [markdown]
# #### Rozwiązanie (dla prowadzącego) — Ćwiczenie 2

# %%
def mc_control_on_policy(env: PModelEnv,
                         gamma: float = 0.99, episodes: int = 50_000,
                         eps: float = 0.2, seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    N = np.zeros((nS, nA), dtype=int)
    rng = np.random.default_rng(seed)
    success_curve = []

    for ep in range(episodes):
        episode = generate_episode_eps_greedy(env, Q, eps, rng, max_steps=max_steps)
        success_curve.append(1 if any(r > 0 for (_, _, r) in episode) else 0)

        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = float(r) + gamma * G
            key = (s, a)
            if key in visited:
                continue
            visited.add(key)
            N[s, a] += 1
            Q[s, a] += (G - Q[s, a]) / N[s, a]

    pi_det = np.argmax(Q, axis=1).astype(int)
    return {"Q": Q, "pi": pi_det, "success_curve": np.asarray(success_curve, dtype=int)}

# %% [markdown]
# ### Demo: MC control (det vs slippery)

# %%
out_det = mc_control_on_policy(env, gamma=0.99, episodes=30_000, eps=0.2, seed=0)
out_slip = mc_control_on_policy(env_slip, gamma=0.99, episodes=60_000, eps=0.2, seed=1)

plt.figure()
plt.plot(moving_average(out_det["success_curve"], 300), label="MC det")
plt.plot(moving_average(out_slip["success_curve"], 300), label="MC slip")
plt.xlabel("epizod (wygładzone 300)")
plt.ylabel("success rate")
plt.title("MC control: det vs slippery")
plt.grid(True); plt.legend(); plt.show()

print("Policy (det):")
action_arrows(out_det["pi"], nrow, ncol, arrows_fl)

# %% [markdown]
# ---
# ## 5. TD(0) prediction: bootstrap zamiast pełnego returnu
#
# TD(0) aktualizuje wartości krokiem:
# `V[s] += alpha * (r + gamma*V[s2] - V[s])`
#
# TODO: WSTAW PSEUDOKOD (Sutton): TD(0) prediction

# %% [markdown]
# ### Ćwiczenie 3 — TD(0) prediction
# Zaimplementuj `td0_prediction` i porównaj z MC prediction dla tej samej polityki.

# %%
def td0_prediction(env: PModelEnv, pi_det: np.ndarray,
                   alpha: float = 0.1, gamma: float = 0.99,
                   episodes: int = 20_000, seed: int = 0, max_steps: int = 200):
    """
    TODO:
    - V = zeros(nS)
    - for ep:
        s = reset
        for t:
            a = pi_det[s]
            step -> s2,r,done
            target = r jeśli done else r + gamma*V[s2]
            V[s] += alpha*(target - V[s])
            break on done
    - return V
    """
    raise NotImplementedError("TODO: td0_prediction")

# %% [markdown]
# #### Rozwiązanie (dla prowadzącego) — Ćwiczenie 3

# %%
def td0_prediction(env: PModelEnv, pi_det: np.ndarray,
                   alpha: float = 0.1, gamma: float = 0.99,
                   episodes: int = 20_000, seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    V = np.zeros(nS, dtype=float)
    rng = np.random.default_rng(seed)

    for ep in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        for _ in range(max_steps):
            a = int(pi_det[s])
            s2, r, done, trunc, _ = env.step(a)
            target = float(r) if (done or trunc) else float(r) + gamma * float(V[s2])
            V[s] += alpha * (target - V[s])
            s = s2
            if done or trunc:
                break
    return V

# %% [markdown]
# ### Demo: TD(0) vs MC (prediction) dla π* (det)

# %%
V_mc = mc_prediction_first_visit(env, pi_star_det, gamma=0.99, episodes=20_000, seed=0)
V_td = td0_prediction(env, pi_star_det, alpha=0.1, gamma=0.99, episodes=20_000, seed=0)

print("V_MC(start) =", float(V_mc[0]))
print("V_TD(start) =", float(V_td[0]))
print("V_DP(start) =", float(v_star_det[0]))

# %% [markdown]
# ---
# ## 6. TD control: SARSA vs Q-learning
#
# - SARSA (on-policy) uczy Q dla polityki wykonywanej (ε-greedy)
# - Q-learning (off-policy) uczy Q dla polityki zachłannej
#
# TODO: WSTAW PSEUDOKOD (Sutton): SARSA i Q-learning

# %% [markdown]
# ### Ćwiczenie 4 — SARSA
# Zaimplementuj SARSA i porównaj z Q-learning (bonus / run&interpret).

# %%
def sarsa(env: PModelEnv,
          alpha: float = 0.5, gamma: float = 0.99, eps: float = 0.2,
          episodes: int = 20_000, seed: int = 0, max_steps: int = 200):
    """
    TODO:
    - Q = zeros((nS,nA))
    - for ep:
        reset -> s
        a = eps-greedy(Q[s])
        loop:
            step -> s2,r,done
            a2 = eps-greedy(Q[s2])
            target = r if done else r + gamma*Q[s2,a2]
            Q[s,a] += alpha*(target - Q[s,a])
            s,a = s2,a2
        success_curve append
    - return dict(Q, pi=argmax(Q), success_curve)
    """
    raise NotImplementedError("TODO: sarsa")

# %% [markdown]
# #### Rozwiązanie (dla prowadzącego) — SARSA

# %%
def sarsa(env: PModelEnv,
          alpha: float = 0.5, gamma: float = 0.99, eps: float = 0.2,
          episodes: int = 20_000, seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    rng = np.random.default_rng(seed)
    success_curve = []

    for ep in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        a = epsilon_greedy_action(Q[s], eps, rng)
        success = 0

        for _ in range(max_steps):
            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                success = 1
            if done or trunc:
                target = float(r)
                Q[s, a] += alpha * (target - Q[s, a])
                break
            a2 = epsilon_greedy_action(Q[s2], eps, rng)
            target = float(r) + gamma * float(Q[s2, a2])
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2

        success_curve.append(success)

    pi_det = np.argmax(Q, axis=1).astype(int)
    return {"Q": Q, "pi": pi_det, "success_curve": np.asarray(success_curve, dtype=int)}

# %% [markdown]
# ### Bonus: Q-learning (run & interpret)
# Kod dostajesz gotowy — uruchom i porównaj z SARSA (zwłaszcza na slippery).

# %%
def q_learning(env: PModelEnv,
               alpha: float = 0.5, gamma: float = 0.99, eps: float = 0.2,
               episodes: int = 20_000, seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    rng = np.random.default_rng(seed)
    success_curve = []

    for ep in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        success = 0
        for _ in range(max_steps):
            a = epsilon_greedy_action(Q[s], eps, rng)
            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                success = 1
            target = float(r) if (done or trunc) else float(r) + gamma * float(np.max(Q[s2]))
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
            if done or trunc:
                break
        success_curve.append(success)

    pi_det = np.argmax(Q, axis=1).astype(int)
    return {"Q": Q, "pi": pi_det, "success_curve": np.asarray(success_curve, dtype=int)}

# %% [markdown]
# ### Demo: SARSA vs Q-learning (det i slippery)

# %%
out_sarsa_det = sarsa(env, episodes=25_000, eps=0.2, alpha=0.5, seed=0)
out_ql_det = q_learning(env, episodes=25_000, eps=0.2, alpha=0.5, seed=1)

out_sarsa_slip = sarsa(env_slip, episodes=80_000, eps=0.2, alpha=0.5, seed=2)
out_ql_slip = q_learning(env_slip, episodes=80_000, eps=0.2, alpha=0.5, seed=3)

plt.figure()
plt.plot(moving_average(out_sarsa_det["success_curve"], 500), label="SARSA det")
plt.plot(moving_average(out_ql_det["success_curve"], 500), label="Q-learn det")
plt.title("TD control (det)")
plt.xlabel("epizod (wygładzone 500)"); plt.ylabel("success rate")
plt.grid(True); plt.legend(); plt.show()

plt.figure()
plt.plot(moving_average(out_sarsa_slip["success_curve"], 1000), label="SARSA slip")
plt.plot(moving_average(out_ql_slip["success_curve"], 1000), label="Q-learn slip")
plt.title("TD control (slippery)")
plt.xlabel("epizod (wygładzone 1000)"); plt.ylabel("success rate")
plt.grid(True); plt.legend(); plt.show()

# %% [markdown]
# ---
# ## 7. Preview: DQN (krótko, run & interpret)
#
# DQN = Q-learning + sieć neuronowa + stabilizacja:
# - replay buffer
# - target network
#
# TODO: WSTAW RYSUNEK (DQN): replay buffer i target network

# %%
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

class DQN(nn.Module):
    def __init__(self, nS, nA):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nS, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, nA),
        )
    def forward(self, x):
        return self.net(x)

def one_hot_state(s: int, nS: int) -> np.ndarray:
    x = np.zeros(nS, dtype=np.float32)
    x[int(s)] = 1.0
    return x

def dqn_train(env: PModelEnv, episodes: int = 2000, gamma: float = 0.99,
              lr: float = 1e-3, batch_size: int = 64,
              buffer_size: int = 50_000, min_buffer: int = 500,
              eps_start: float = 1.0, eps_end: float = 0.05, eps_decay: float = 0.995,
              target_update: int = 200, max_steps: int = 200, seed: int = 0):
    rng = np.random.default_rng(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    nS = env.observation_space.n
    nA = env.action_space.n

    q = DQN(nS, nA)
    q_tgt = DQN(nS, nA)
    q_tgt.load_state_dict(q.state_dict())
    q_tgt.eval()

    opt = optim.Adam(q.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    buffer = deque(maxlen=buffer_size)
    success_curve = []
    loss_curve = []

    eps = eps_start
    global_step = 0

    for ep in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        ep_success = 0

        for _ in range(max_steps):
            if rng.random() < eps:
                a = int(rng.integers(0, nA))
            else:
                with torch.no_grad():
                    qs = q(torch.tensor(one_hot_state(s, nS)).unsqueeze(0))
                    a = int(torch.argmax(qs, dim=1).item())

            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                ep_success = 1

            buffer.append((s, a, r, s2, done or trunc))
            s = s2
            global_step += 1

            if len(buffer) >= min_buffer:
                batch = random.sample(buffer, batch_size)
                S = torch.tensor(np.stack([one_hot_state(b[0], nS) for b in batch]))
                A = torch.tensor([b[1] for b in batch], dtype=torch.long)
                R = torch.tensor([b[2] for b in batch], dtype=torch.float32)
                S2 = torch.tensor(np.stack([one_hot_state(b[3], nS) for b in batch]))
                D = torch.tensor([b[4] for b in batch], dtype=torch.float32)

                q_sa = q(S).gather(1, A.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_next = q_tgt(S2).max(dim=1).values
                    target = R + (1.0 - D) * gamma * q_next

                loss = loss_fn(q_sa, target)
                opt.zero_grad()
                loss.backward()
                opt.step()

                loss_curve.append(float(loss.item()))

                if global_step % target_update == 0:
                    q_tgt.load_state_dict(q.state_dict())

            if done or trunc:
                break

        success_curve.append(ep_success)
        eps = max(eps_end, eps * eps_decay)

    pi_det = np.zeros(nS, dtype=int)
    with torch.no_grad():
        for s in range(nS):
            qs = q(torch.tensor(one_hot_state(s, nS)).unsqueeze(0))
            pi_det[s] = int(torch.argmax(qs, dim=1).item())

    return {"pi": pi_det, "success_curve": np.asarray(success_curve), "loss_curve": np.asarray(loss_curve)}

out_dqn = dqn_train(env, episodes=2500, gamma=0.99, seed=0)

plt.figure()
plt.plot(moving_average(out_dqn["success_curve"], 200))
plt.xlabel("epizod (wygładzone 200)")
plt.ylabel("success rate")
plt.title("DQN preview (FrozenLake det)")
plt.grid(True)
plt.show()

plt.figure()
if len(out_dqn["loss_curve"]) > 0:
    plt.plot(moving_average(out_dqn["loss_curve"], 200))
plt.xlabel("update (wygładzone 200)")
plt.ylabel("loss")
plt.title("DQN loss (moving average)")
plt.grid(True)
plt.show()

print("DQN learned policy (det):")
action_arrows(out_dqn["pi"], nrow, ncol, arrows_fl)

# %% [markdown]
# ---
# ## 8. Pytania do wniosków (oddanie)
# Napisz 8–12 zdań:
# 1) MC vs TD: kto szybciej daje sensowne `V(start)` przy małej liczbie epizodów i dlaczego?  
# 2) SARSA vs Q-learning na slippery: kto jest „ostrożniejszy” i dlaczego (on/off-policy)?  
# 3) DQN: po co replay buffer i target network? Jakie problemy stabilności rozwiązują?  
#
# To domyka intuicję przed przejściem do **policy gradient / actor-critic / PPO**.


TypeError: 'type' object is not subscriptable